In [39]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import Image, HTML

In [50]:
df = pd.read_csv('../Datasets/amazon_final.csv')

In [51]:
def detectar_intrusos_selectivo(df, umbral_inferior=0.1, umbral_superior=10.0):
    """
    Limpia solo categorías con alta disparidad de precios, excluyendo 'Other Electronics'.
    """
    # 1. Crear backup de seguridad
    df['category_backup'] = df['product_category']
    
    # 2. Identificar categorías candidatas (excluyendo 'Other Electronics')
    # Calculamos la relación entre el percentil 95 y el percentil 5 para ver la dispersión
    stats = df.groupby('product_category')['original_price'].agg(['median', lambda x: x.quantile(0.95) / x.quantile(0.05)])
    stats.columns = ['mediana', 'dispersion_ratio']
    
    # Definimos qué categorías "merecen" limpieza: 
    # Aquellas donde el 5% más caro es, por ejemplo, 20 veces más caro que el 5% más barato
    categorias_a_limpiar = stats[(stats['dispersion_ratio'] > 20) & 
                                 (stats.index != 'Other Electronics')].index.tolist()
    
    print(f"Categorías seleccionadas para limpieza por alta disparidad: {categorias_a_limpiar}")

    # 3. Aplicar la lógica de intrusos SOLO en esas categorías
    df['es_intruso'] = False
    
    for cat in categorias_a_limpiar:
        mediana_cat = stats.loc[cat, 'mediana']
        
        condicion = (df['product_category'] == cat) & (
            (df['original_price'] < (mediana_cat * umbral_inferior)) | 
            (df['original_price'] > (mediana_cat * umbral_superior))
        )
        df.loc[condicion, 'es_intruso'] = True

    # 4. Reasignar y mostrar resultados
    num_intrusos = df['es_intruso'].sum()
    if num_intrusos > 0:
        print(f"\n✅ Se han detectado {num_intrusos} intrusos en las categorías seleccionadas.")
        print(df[df['es_intruso']].groupby('category_backup').size())
        
        df.loc[df['es_intruso'], 'product_category'] = 'category_misc'
    else:
        print("No se detectaron intrusos significativos en las categorías objetivo.")

    # Limpiamos columna auxiliar
    df.drop(columns=['es_intruso'], inplace=True)
    
    return df

# Ejecutar la limpieza quirúrgica
df_clean = detectar_intrusos_selectivo(df)

Categorías seleccionadas para limpieza por alta disparidad: ['Audio, Sound & Recording Gear', 'Cameras & Photography', 'Chargers, Adapters & Cables', 'Computer Peripherals', 'Laptops', 'Mobile Cell Phones & Smartphones', 'Networking', 'Office Supplies, Ink & Toner', 'PC Components', 'Power & Batteries', 'Printers & Scanners (Hardware)', 'Small Gadget Accessories (Cases & Protectors)', 'Smart Home & Security', 'Storage & Memory Cards', 'TV & Video Displays', 'Tablets & E-readers (DEVICES ONLY)', 'Video Game Consoles & Virtual Reality', 'Wearables']

✅ Se han detectado 389 intrusos en las categorías seleccionadas.
category_backup
Audio, Sound & Recording Gear                    30
Cameras & Photography                            53
Chargers, Adapters & Cables                      15
Computer Peripherals                             21
Laptops                                          51
Mobile Cell Phones & Smartphones                  1
Networking                                        6


In [52]:
from collections import Counter
import re

# 1. Filtramos los títulos de los 389 intrusos
titulos_intrusos = df_clean[df_clean['product_category'] == 'category_misc']['product_title'].astype(str)

# 2. Limpieza rápida: todo a minúsculas y solo palabras de más de 3 letras
texto_completo = " ".join(titulos_intrusos).lower()
palabras = re.findall(r'\b\w{4,}\b', texto_completo) # Filtra palabras cortas como 'de', 'el', 'la'

# 3. Lista de palabras a ignorar (añade las que veas que ensucian el resultado)
ignorar = {'black', 'pack', 'with', 'para', 'color', 'compatible', 'amazon', 'basics'}
palabras_limpias = [w for w in palabras if w not in ignorar]

# 4. Sacamos las 30 más frecuentes
top_keywords = Counter(palabras_limpias).most_common(30)

print("🔍 TOP 30 KEYWORDS EN CATEGORY_MISC:")
for palabra, frec in top_keywords:
    print(f"{palabra}: {frec}")

🔍 TOP 30 KEYWORDS EN CATEGORY_MISC:
laptop: 66
camera: 50
inch: 48
lens: 42
cable: 40
hdmi: 37
gaming: 34
card: 31
battery: 29
sony: 27
video: 27
full: 27
nvidia: 26
case: 25
mirrorless: 25
frame: 24
ugreen: 23
ultra: 23
computer: 22
high: 22
32gb: 22
canon: 21
display: 21
macbook: 21
core: 21
smart: 21
remote: 20
mini: 20
ddr5: 19
series: 18


In [53]:
def reclasificacion_final_tfm(df):
    # 1. Diccionario de Mapeo: Keyword -> Categoría Destino Oficial
    mapeo_categorias = {
        'Laptops': ['laptop', 'macbook', 'notebook', 'chromebook'],
        'PC Components': ['nvidia', 'rtx', 'gtx', 'ddr5', 'core', 'intel', 'ryzen', 'ssd', 'motherboard', '32gb'],
        'Cameras & Photography': ['camera', 'mirrorless', 'lens', 'dslr', 'canon', 'sony', 'fujifilm', 'nikon'],
        'Chargers, Adapters & Cables': ['cable', 'hdmi', 'adapter', 'ugreen', 'charger', 'usb c', 'power cord'],
        'Small Gadget Accessories (Cases & Protectors)': ['case', 'sleeve', 'protector', 'shell', 'mount', 'stand'],
        'Power & Batteries': ['battery', 'ups', 'power bank', 'power station'],
        'Storage & Memory Cards': ['card', 'sd card', 'microsd', 'flash drive'],
        'Audio, Sound & Recording Gear': ['earbuds', 'headphones', 'headset', 'speaker', 'microphone'],
        'TV & Video Displays': ['remote', 'monitor', 'screen', 'display']
    }

    # 2. Backup de seguridad
    if 'category_backup' not in df.columns:
        df['category_backup'] = df['product_category']

    def refinar(row):
        # Solo intentamos reclasificar si es un intruso actual o está en misc
        if row['product_category'] != 'category_misc':
            return row['product_category']
        
        titulo = str(row['product_title']).lower()
        precio = row['original_price']
        
        # A. Intentar rescatar por palabras clave
        for cat_oficial, keywords in mapeo_categorias.items():
            if any(key in titulo for key in keywords):
                # Si es una categoría cara pero el producto es muy barato, 
                # lo mandamos a accesorios en lugar de a la categoría principal
                if cat_oficial in ['Laptops', 'PC Components', 'Cameras & Photography'] and precio < 40:
                    return 'Small Gadget Accessories (Cases & Protectors)'
                return cat_oficial
        
        # B. Rescate por valor: si es caro y no sabemos qué es, devolvemos a su origen
        if precio > 150:
            return row['category_backup']
            
        return 'category_misc' # Si nada coincide, se queda en misc

    df['product_category'] = df.apply(refinar, axis=1)
    
    # 3. Resumen de éxito
    print("✅ Reclasificación completada.")
    print(df['product_category'].value_counts())
    return df

df_clean = reclasificacion_final_tfm(df_clean)

✅ Reclasificación completada.
product_category
Office Supplies, Ink & Toner                     1094
Computer Peripherals                              739
Audio, Sound & Recording Gear                     708
Small Gadget Accessories (Cases & Protectors)     676
PC Components                                     622
TV & Video Displays                               568
Chargers, Adapters & Cables                       512
Laptops                                           437
Cameras & Photography                             427
Wearables                                         406
Storage & Memory Cards                            310
Power & Batteries                                 272
Networking                                        172
Smart Home & Security                             168
Printers & Scanners (Hardware)                    159
Mobile Cell Phones & Smartphones                  142
Other Electronics                                 131
Tablets & E-readers (DEVICES ONLY) 

In [54]:
df_clean.columns

Index(['product_title', 'product_rating', 'total_reviews',
       'purchased_last_month', 'original_price', 'is_sponsored', 'coupon',
       'buy_box_availability', 'sustainability_tags', 'product_image_url',
       'has_coupon', 'discount_percentage', 'badge_amazons',
       'badge_best_seller', 'badge_ends_in', 'badge_limited_time_deal',
       'badge_no_badge', 'badge_save_xpct', 'product_category',
       'log_purchased_last_month', 'log_original_price', 'log_total_reviews',
       'category_backup'],
      dtype='object')

In [56]:
# Filtramos filas donde la categoría actual es distinta a la de backup
cambios = df_clean[df_clean['product_category'] != df_clean['category_backup']]

# Seleccionamos columnas clave para auditar
auditoria = cambios

print(f"Se han reclasificado {len(auditoria)} productos.")
display(auditoria.sample(30)) # Mostramos los primeros 30 para revisar

Se han reclasificado 238 productos.


,product_title,product_rating,total_reviews,purchased_last_month,original_price,is_sponsored,coupon,buy_box_availability,sustainability_tags,product_image_url,...,badge_best_seller,badge_ends_in,badge_limited_time_deal,badge_no_badge,badge_save_xpct,product_category,log_purchased_last_month,log_original_price,log_total_reviews,category_backup
3105,"DJI Osmo Pocket 3 Creator Combo Gimbal Stabilizer with 1′′ CMOS & 4K/120fps Video, 3-Axis Stabilization Bundle with 128GB Memory Card, Gripster Flexible Spider Tripod, 2 YR CPS Warranty + More",5.0,2.0,100,1295.00,0,No Coupon,1,0,https://m.media-amazon.com/images/I/81gAjW5pdQL._AC_UL320_.jpg,...,0,0,0,1,0,Storage & Memory Cards,4.615121,7.167038,1.098612,"Audio, Sound & Recording Gear"
2634,"BAGSMART Laptop Sleeve Bag Compatible with MacBook Air/Pro, 13-13.3 inch Notebook, Compatible with MacBook Pro 14 Inch, MacBook Air M2 Sleeve 13 Inch, Repellent Protective Case with Pocket, Pink",4.8,5368.0,1000,11.59,0,No Coupon,1,0,https://m.media-amazon.com/images/I/71mmykcBE2S._AC_UL320_.jpg,...,0,0,0,1,0,Small Gadget Accessories (Cases & Protectors),6.908755,2.532903,8.588397,Laptops
6570,"Toshiba 500GB 2.5-inch SATA Laptop Hard Drive (5400rpm, 8MB Cache) MQ01ABD050, Mechanical Hard Disk",4.5,1467.0,300,18.19,0,No Coupon,1,0,https://m.media-amazon.com/images/I/81p2buxQ-NL._AC_UL320_.jpg,...,0,0,0,1,0,Small Gadget Accessories (Cases & Protectors),5.707110,2.954389,7.291656,Laptops
2157,ASURION 2 Year Wearables Protection Plan ($50 - $59.99),4.2,110.0,0,7.99,0,No Coupon,0,0,https://m.media-amazon.com/images/I/51Xr76m2WLL._AC_UL320_.jpg,...,0,0,0,1,0,category_misc,0.000000,2.196113,4.709530,Wearables
634,"UGREEN USB A to USB B Printer Cable 5ft - High-Speed for HP, Canon, Brother, Samsung, Dell, Epson, Lexmark, Xerox, and More",4.7,44709.0,8000,5.69,0,No Coupon,1,0,https://m.media-amazon.com/images/I/61NeoMyErcL._AC_UL320_.jpg,...,0,0,0,1,0,Small Gadget Accessories (Cases & Protectors),8.987322,1.900614,10.707952,Printers & Scanners (Hardware)
6104,Yamaha Viola Bow FC5,4.6,5308.0,400,13.50,0,No Coupon,0,0,https://m.media-amazon.com/images/I/71dZJfQoIaL._AC_UL320_.jpg,...,0,0,0,1,0,category_misc,5.993961,2.674149,8.577159,PC Components
1544,"Apple Late 2020 MacBook Pro with Apple M1 Chip, 13-inch, 8GB RAM, 512GB SSD, Space Gray (Renewed)",4.4,369.0,400,1019.00,0,No Coupon,0,0,https://m.media-amazon.com/images/I/71an9eiBxpL._AC_UL320_.jpg,...,0,0,0,1,0,Laptops,5.993961,6.927558,5.913503,Storage & Memory Cards
6923,"BenQ W4100i 4K HDR Smart Home Theater Projector, 3200 Lumens, 100% DCI-P3, Rec.709, Factory-Calibrated, Android TV with Netflix, 4-Way Lens Shift, HDR10+ & HLG Support, LED Long Lifespan",4.4,7.0,0,2999.00,0,No Coupon,0,0,https://m.media-amazon.com/images/I/515sOBZYo3L._AC_UL320_.jpg,...,0,0,0,1,0,Cameras & Photography,0.000000,8.006368,2.079442,TV & Video Displays
5727,Sony FE 50-150 F2 GM,5.0,4.0,0,3898.00,0,No Coupon,0,0,https://m.media-amazon.com/images/I/71eGPt2alsL._AC_UL320_.jpg,...,0,0,0,1,0,Cameras & Photography,0.000000,8.268475,1.609438,TV & Video Displays
4241,"Highland Sticky Notes, 1.5 x 2 Inches, Yellow, 12 Pack (6539)",4.5,1580.0,1000,5.51,0,No Coupon,1,0,https://m.media-amazon.com/images/I/61+YAfk1+bL._AC_UL320_.jpg,...,0,0,0,1,0,category_misc,6.908755,1.873339,7.365813,"Office Supplies, Ink & Toner"


In [57]:
# 1. Creamos una copia por seguridad
df_clean_final = df_clean.copy()

# 2. Aseguramos que el índice sea el mismo (si no lo has tocado, ya debería serlo)
# Esto "pisa" los datos de df_clean con los de auditoria
df_clean_final.update(auditoria)

# 3. ¡Importante! update a veces convierte columnas a 'object' 
# Vamos a asegurar que los precios siguen siendo numéricos
df_clean_final['log_original_price'] = pd.to_numeric(df_clean_final['log_original_price'])
df_clean_final['original_price'] = pd.to_numeric(df_clean_final['original_price'])

In [60]:
df_clean_final.drop(columns=['category_backup'], inplace=True)

In [61]:
df_clean_final.to_csv('../Datasets/auditoria1.csv', index=False)

In [59]:
df_clean_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7717 entries, 0 to 7716
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   product_title             7717 non-null   object 
 1   product_rating            7717 non-null   float64
 2   total_reviews             7717 non-null   float64
 3   purchased_last_month      7717 non-null   int64  
 4   original_price            7717 non-null   float64
 5   is_sponsored              7717 non-null   int64  
 6   coupon                    7717 non-null   object 
 7   buy_box_availability      7717 non-null   int64  
 8   sustainability_tags       7717 non-null   int64  
 9   product_image_url         7717 non-null   object 
 10  has_coupon                7717 non-null   int64  
 11  discount_percentage       7717 non-null   float64
 12  badge_amazons             7717 non-null   int64  
 13  badge_best_seller         7717 non-null   int64  
 14  badge_en